# Tarea 03: Generación de imágenes con autoencoders
### PyTorch Lightning · Hydra · Weights & Biases

**Curso:** Inteligencia Artificial — Escuela de Ingeniería en Computación, Instituto Tecnológico de Costa Rica

**Integrantes:**
- Sebastián Chacón - 2023221908
- Daniel Pulido - 2019196502
- Sebastián Muñoz - 2023090681

**Objetivo:** comparar un *Variational Autoencoder* (VAE) y un autoencoder U-Net (con *skip connections*) en la reconstrucción de imágenes de MVTec AD, entrenando cada uno con cuatro funciones de pérdida (L1, L2, SSIM y SSIM + L1) y analizando su capacidad para distinguir imágenes normales de imágenes con anomalías.

> Todas las figuras de resultados de este notebook se generan y registran en Weights & Biases y luego se insertan aquí (descargándolas con la API de W&B).

## Contenido
1. Configuración del entorno
2. Dataset (MVTec AD)
3. Configuración de experimentos con Hydra
4. Modelos
5. Funciones de pérdida
6. Entrenamiento
7. Seguimiento de los entrenamientos en W&B
8. Evaluación del modelo
9. Comparación entre configuraciones
10. Conclusiones

## 1. Configuración del entorno

### 1.1 Entorno virtual (`.venv`)
El proyecto usa un entorno virtual propio con **Python 3.11** (Hydra y Lightning son más estables en 3.11 que en 3.14). Se crea **una sola vez** desde una terminal en la raíz del repositorio, antes de abrir el notebook, porque el notebook tiene que ejecutarse *dentro* de ese entorno:

```powershell
py -3.11 -m venv .venv
.venv\Scripts\python -m pip install --upgrade pip
# PyTorch con CUDA (GPU NVIDIA). Sin GPU: quitar --index-url para instalar la versión CPU
.venv\Scripts\python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
.venv\Scripts\python -m pip install -r requirements.txt
# Registrar el entorno como kernel de Jupyter
.venv\Scripts\python -m ipykernel install --user --name autoencoder --display-name "Python (.venv Autoencoder)"
```

Luego, en VS Code: **Select Kernel → Python Environments → `.venv`** (o el kernel *Python (.venv Autoencoder)* en Jupyter).
La siguiente celda verifica que el notebook esté corriendo con el Python del `.venv`, imprime las versiones y si hay GPU.

### 1.2 Imports, versiones y directorio de trabajo
El notebook vive en `notebooks/`, pero `train.py`, `conf/` y `data/` usan rutas relativas a la raíz del repositorio, así que se cambia el directorio de trabajo a la raíz.

In [ ]:
# Imports, versiones de librerías, GPU y directorio de trabajo
import os
import sys
from pathlib import Path

# The notebook lives in notebooks/; train.py, conf/ and data/ are relative to the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import hydra
import lightning as L
import pandas as pd
import plotly
import torch
import torchmetrics
import torchvision
import wandb
from IPython.display import Image, display

print("Raíz del repo:", ROOT)
print("Python       :", sys.executable)
if ".venv" not in sys.executable:
    print("ADVERTENCIA: el kernel no es el del .venv (ver 1.1)")
for name, module in [("torch", torch), ("torchvision", torchvision), ("lightning", L), ("hydra-core", hydra),
                     ("wandb", wandb), ("torchmetrics", torchmetrics), ("plotly", plotly), ("pandas", pd)]:
    print(f"{name:<13}: {module.__version__}")
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible (CPU)")

### 1.3 Weights & Biases
Todos los experimentos se registran en W&B (proyecto `Autoencoder`, definido en `conf/logger/wandb.yaml`). Cada corrida se llama `<modelo>-<pérdida>` (por ejemplo `vae-ssim_l1`), se agrupa por modelo y guarda la configuración completa de Hydra. Durante el entrenamiento se registran:

| Clave en W&B | Contenido |
|---|---|
| `train/loss`, `val/loss` | pérdida de entrenamiento y validación por época (la función de pérdida de la corrida + β·KL en el VAE) |
| `train/l1`, `val/mse`, `val/ssim`, ... | métricas comunes a todas las corridas, para comparar en la misma escala |
| `val/reconstructions` | 16 imágenes de validación: originales vs reconstruidas (cada 5 épocas) |
| `val/tsne` | t-SNE del vector latente del set de validación (cada 10 épocas) |
| `test/reconstructions` | 8 imágenes buenas + 8 defectuosas del set de prueba con su reconstrucción y mapa de error |
| `test/tsne` | t-SNE del set de prueba (good vs defective) |
| `test/scores` | tabla con el error de reconstrucción de cada imagen de prueba (para los histogramas) |
| `test/auroc_*` | AUROC del error de reconstrucción como detector de anomalías (global y por clase) |

**Comandos útiles** (terminal con el `.venv` activo, o con `!` desde una celda):

| Comando | Para qué |
|---|---|
| `wandb login` | guarda la API key (https://wandb.ai/authorize) en el equipo; equivale a `wandb.login()` |
| `wandb offline` / `wandb online` | desactiva / reactiva la sincronización para todo el directorio |
| `wandb sync <carpeta>/wandb/offline-run-*` | sube a W&B una corrida que se hizo en modo offline |
| `python train.py logger.wandb.mode=disabled` | entrena sin registrar nada en W&B |
| `python train.py logger.wandb.entity=<equipo>` | registra en la cuenta de un equipo en vez de la personal |

La primera vez, `wandb.login()` pide la API key; después queda guardada y los entrenamientos lanzados desde este notebook (sección 6) la reutilizan.

In [ ]:
# Inicio de sesión en W&B
WANDB_PROJECT = "Autoencoder"  # same as conf/logger/wandb.yaml
WANDB_ENTITY = None                     # None = default W&B account (or the team name)

wandb.login()
print("Cuenta de W&B:", wandb.Api().default_entity)

Funciones auxiliares para que las figuras que se construyen en el notebook también pasen por W&B: se registran en una corrida aparte (`job_type="analysis"`, no se mezcla con los entrenamientos) y luego se muestran aquí.

In [ ]:
# Funciones auxiliares para registrar figuras en W&B y mostrarlas
from PIL import Image as PILImage


def start_notebook_run(name):
    # W&B run where the notebook logs its own figures (kept apart from the training runs)
    wandb.finish()
    return wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name, job_type="analysis")


def log_and_show(run, key, fig=None, image=None, caption=None):
    # Log a plotly figure or an image to W&B and display it in the notebook
    if fig is not None:
        run.log({key: wandb.Plotly(fig)})
        fig.show()
    if image is not None:
        img = PILImage.open(image) if isinstance(image, (str, Path)) else image
        run.log({key: wandb.Image(img, caption=caption)})
        if caption:
            print(caption)
        display(img)


def show_image(path, title=None):
    # Display an image downloaded from W&B
    if title:
        print(title)
    display(Image(filename=str(path)))

## 2. Dataset (MVTec AD)
*MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection* contiene imágenes industriales de 15 objetos/texturas. El conjunto de **entrenamiento** solo tiene imágenes sin defectos (`good`), y el de **prueba** mezcla imágenes buenas con distintos tipos de defectos. Por eso es un problema de detección de anomalías **no supervisada**: el autoencoder aprende a reconstruir lo "normal" y se espera que reconstruya peor lo defectuoso.

Para reducir el tiempo de entrenamiento se usan solo 4 clases: **cable, capsule, screw y transistor**.

### 2.1 Estructura y cantidad de imágenes
Se descargaron los archivos `cable.tar.xz`, `capsule.tar.xz`, `screw.tar.xz` y `transistor.tar.xz` de la página oficial y se extrajeron en `data/mvtec_ad/` (la carpeta `data/` no se sube a git):

```bash
mkdir -p data/mvtec_ad
for c in cable capsule screw transistor; do tar -xJf ~/Downloads/$c.tar.xz -C data/mvtec_ad; done
```

Estructura resultante: `data/mvtec_ad/<clase>/train/good/`, `data/mvtec_ad/<clase>/test/<good|tipo_de_defecto>/` y `ground_truth/` (máscaras, no se usan). La siguiente celda verifica la estructura y cuenta las imágenes por clase y tipo de defecto.

In [ ]:
# Verificación del dataset y conteo de imágenes por clase/defecto
from collections import Counter

from src.data import list_samples

DATA_ROOT = ROOT / "data" / "mvtec_ad"
CLASSES = ["cable", "capsule", "screw", "transistor"]

missing = [c for c in CLASSES if not (DATA_ROOT / c / "train" / "good").is_dir()]
assert not missing, f"Faltan clases en {DATA_ROOT}: {missing} (ver 2.1)"

train_samples = list_samples(DATA_ROOT, CLASSES, "train")
test_samples = list_samples(DATA_ROOT, CLASSES, "test")

test_df = pd.DataFrame([(c, d) for _, c, d in test_samples], columns=["clase", "defecto"])
summary = pd.DataFrame({
    "train (good)": pd.Series(Counter(c for _, c, _ in train_samples)),
    "test good": test_df[test_df.defecto == "good"].clase.value_counts(),
    "test defectuosas": test_df[test_df.defecto != "good"].clase.value_counts(),
    "tipos de defecto": test_df[test_df.defecto != "good"].groupby("clase").defecto.nunique(),
}).loc[CLASSES]
summary.loc["total"] = summary.sum()
display(summary)

print("Imágenes de prueba por tipo de defecto:")
display(test_df.groupby(["clase", "defecto"]).size().rename("n").to_frame())

### 2.2 Preprocesamiento: 128x128 con 3 canales
Las imágenes originales son de alta resolución (entre 1000x1000 y 1024x1024; *screw* además está en escala de grises). Al cargarlas (`src/data.py`):
- se convierten a **RGB** (screw pasa de 1 a 3 canales repitiendo el canal gris),
- se redimensionan a **128x128** con interpolación bilineal,
- se guardan en memoria como `uint8` y se convierten a `float` en **[0, 1]** al pedir un batch. El rango [0, 1] coincide con la salida `Sigmoid` de ambos modelos y con el `data_range=1` de SSIM.

No se usa *data augmentation*: el objetivo es aprender la apariencia "normal" de cada objeto, y transformaciones como rotaciones podrían parecerse a defectos (p. ej. `misplaced` en transistor).

La configuración del dataset se lee del archivo de Hydra `conf/data/mvtec.yaml`.

In [ ]:
# Carga del DataModule y forma de un batch
from omegaconf import OmegaConf

from src.data import MVTecDataModule

data_cfg = OmegaConf.load(ROOT / "conf" / "data" / "mvtec.yaml")
print(OmegaConf.to_yaml(data_cfg))

dm = MVTecDataModule(**data_cfg)
dm.setup()  # loads and resizes every image once (can take a minute)

batch = next(iter(dm.train_dataloader()))
x = batch["image"]
print("Batch:", tuple(x.shape), x.dtype, f"rango [{x.min():.2f}, {x.max():.2f}]")
print("Clases en el batch:", Counter(batch["category"]))

### 2.3 División entrenamiento / validación / prueba
- **Entrenamiento y validación** salen solo de `train/good`: se separa un 15 % (`val_split`) **de cada clase**, para que la validación contenga los 4 objetos. La validación mide qué tan bien se reconstruyen imágenes normales que el modelo no vio.
- **Prueba** es la carpeta `test/` completa: imágenes buenas y defectuosas. Solo se usa al final, con el mejor checkpoint (menor `val/loss`).

In [ ]:
# Tamaños de los splits y ejemplos del set de prueba (vía W&B)
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from src.callbacks import spread_indices

splits = {"train": dm.train_dataset, "val": dm.val_dataset, "test": dm.test_dataset}
sizes = pd.DataFrame({name: pd.Series(Counter(ds.categories)) for name, ds in splits.items()}).loc[CLASSES]
sizes.loc["total"] = sizes.sum()
display(sizes)
print("Test good / defective:", Counter("good" if l == 0 else "defective" for l in dm.test_dataset.labels))

# Examples: first row good, second row defective (2 per class), logged to W&B and shown here
ds = dm.test_dataset
good = [i for i, l in enumerate(ds.labels) if l == 0]
bad = [i for i, l in enumerate(ds.labels) if l == 1]
idx = spread_indices(good, ds.categories, 8) + spread_indices(bad, ds.categories, 8)
grid = make_grid(ds.get_images(idx), nrow=8, padding=2, pad_value=1.0)

dataset_run = start_notebook_run("notebook-dataset")
log_and_show(dataset_run, "dataset/test_examples", image=to_pil_image(grid),
             caption="Fila 1: good. Fila 2: defectuosas (" + ", ".join(ds.defects[i] for i in idx[8:]) + ")")

## 3. Configuración de experimentos con Hydra
Toda la configuración vive en `conf/` y se separa por responsabilidad, de modo que cambiar el modelo, la pérdida o el entrenamiento es solo elegir otro archivo o sobrescribir un valor:

```
conf/
├── config.yaml        # compone un archivo de cada grupo + semilla, nombre de la corrida y el sweep
├── data/mvtec.yaml    # ruta, clases, tamaño de imagen, batch size, split de validación
├── model/
│   ├── vae.yaml       # red (_target_: src.models.vae.VAE), latent_dim (z), β del KL, optimizador
│   └── u-net.yaml     # red (_target_: src.models.unet.UNet), latent_dim, optimizador
├── loss/              # l1.yaml, l2.yaml, ssim.yaml, ssim_l1.yaml (_target_ de cada pérdida)
├── trainer/default.yaml   # argumentos del Trainer de Lightning (épocas, GPU, precisión)
└── logger/wandb.yaml      # WandbLogger (proyecto, nombre, tags) y frecuencia de las figuras
```

- La lista `defaults` de `config.yaml` elige la opción de cada grupo (`model: vae`, `loss: l1`, ...).
- Los bloques con `_target_` se construyen con `hydra.utils.instantiate`, así `train.py` no tiene código específico de cada modelo o pérdida.
- Desde la línea de comandos se puede cambiar un grupo (`model=u-net`), un valor (`model.net.latent_dim=64`, `trainer.max_epochs=50`) o barrer varias opciones con `-m` (multirun).

La celda siguiente compone la configuración con la API de Hydra, igual que lo hace `train.py`.

In [ ]:
# Composición de la configuración con Hydra
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

CONF_DIR = str(ROOT / "conf")


def load_cfg(overrides=()):
    # Compose conf/config.yaml with optional overrides, exactly as `python train.py <overrides>` would
    with initialize_config_dir(config_dir=CONF_DIR, version_base=None):
        return compose(config_name="config", overrides=list(overrides))


cfg = load_cfg()
print(OmegaConf.to_yaml(cfg))

Ejemplo de *overrides*: U-Net con SSIM + L1 y espacio latente de 64. Equivale a `python train.py model=u-net loss=ssim_l1 model.net.latent_dim=64`.

In [ ]:
# Ejemplo de overrides de Hydra
cfg_example = load_cfg(["model=u-net", "loss=ssim_l1", "model.net.latent_dim=64"])
print(OmegaConf.to_yaml(cfg_example.model))
print(OmegaConf.to_yaml(cfg_example.loss))

## 4. Modelos
Ambos modelos reciben imágenes de 3x128x128, devuelven la reconstrucción en [0, 1] (`Sigmoid`) y un vector latente `z` que se usa para el t-SNE. Se entrenan con el mismo `LightningModule` (`src/module.py`), que recibe la red, la pérdida y el optimizador ya construidos por Hydra.

### 4.1 Variational Autoencoder (VAE)
`src/models/vae.py`
- **Encoder:** 4 convoluciones con stride 2 (128 → 64 → 32 → 16 → 8) con BatchNorm y LeakyReLU; el mapa de 256x8x8 se aplana y dos capas lineales producen la media $\mu$ y el log-varianza $\log\sigma^2$ de $q(z|x)$, con $z \in \mathbb{R}^{128}$ (`latent_dim`).
- **Truco de reparametrización:** $z = \mu + \sigma \odot \varepsilon$, $\varepsilon \sim \mathcal{N}(0, I)$, para que el muestreo sea diferenciable. En evaluación se usa $z = \mu$.
- **Decoder:** capa lineal a 256x8x8 y 4 convoluciones transpuestas de vuelta a 128x128.
- **Pérdida:** $\mathcal{L} = \mathcal{L}_{rec}(x, \hat{x}) + \beta \, KL\left(q(z|x) \,\|\, \mathcal{N}(0, I)\right)$ con $\beta = 0.001$. El término KL ordena el espacio latente; un $\beta$ pequeño evita que domine sobre la reconstrucción (imágenes borrosas).

Toda la información tiene que pasar por el cuello de botella de 128 números, por lo que el VAE no puede copiar detalles finos: eso es lo que debería hacer que los defectos no se reconstruyan.

In [ ]:
# Construcción y resumen del VAE
from lightning.pytorch.utilities.model_summary import ModelSummary

from src.module import AutoencoderModule


def build_module(cfg):
    # Same construction as train.py: network, loss and optimizer come from the Hydra config
    return AutoencoderModule(
        net=instantiate(cfg.model.net),
        loss_fn=instantiate(cfg.loss.fn),
        optimizer=instantiate(cfg.model.optimizer),
        beta=cfg.model.beta,
    )


def describe(module, x):
    print(ModelSummary(module, max_depth=2))
    with torch.no_grad():
        out = module.eval()(x)
    for key, value in out.items():
        print(f"{key:<7}: {None if value is None else tuple(value.shape)}")


vae_module = build_module(load_cfg(["model=vae"]))
describe(vae_module, x[:4])

### 4.2 Autoencoder U-Net
`src/models/unet.py`
- **Encoder:** bloques `DoubleConv` (2 × [conv 3x3 + BatchNorm + ReLU]) con MaxPool entre niveles: 32 → 64 → 128 → 256 canales y un cuello de botella de 128x8x8.
- **Decoder:** convoluciones transpuestas que duplican la resolución y, en cada nivel, **concatenan el mapa del encoder de la misma resolución (*skip connection*)** antes de otro `DoubleConv`.
- **Vector latente:** el promedio global del cuello de botella (128 valores) se usa como `z` para el t-SNE. No hay término KL ($\beta = 0$).

Las *skip connections* dan al decoder acceso directo a los detalles de alta resolución, así que la U-Net reconstruye mucho más nítido que el VAE. El riesgo para detección de anomalías es que también puede "copiar" los defectos a través de las skips, reduciendo la diferencia de error entre imágenes buenas y defectuosas. Esta es una de las hipótesis a revisar en los resultados.

In [ ]:
# Construcción y resumen de la U-Net
unet_module = build_module(load_cfg(["model=u-net"]))
describe(unet_module, x[:4])

## 5. Funciones de pérdida
Con $x$ la imagen original y $\hat{x}$ la reconstrucción (ambas en [0, 1]):

| Config | Definición | Qué penaliza |
|---|---|---|
| `l1` | $\frac{1}{N}\sum \lvert x - \hat{x} \rvert$ | error absoluto por píxel; robusta a valores atípicos, tiende a imágenes más nítidas que L2 |
| `l2` | $\frac{1}{N}\sum (x - \hat{x})^2$ | error cuadrático; castiga mucho los errores grandes y tolera los pequeños, lo que suele dar reconstrucciones borrosas |
| `ssim` | $1 - \mathrm{SSIM}(x, \hat{x})$ | diferencias de estructura, contraste y luminancia en ventanas locales (gaussianas 11x11); poco sensible a cambios uniformes de brillo/color |
| `ssim_l1` | $\alpha (1 - \mathrm{SSIM}) + (1 - \alpha) \, L1$, $\alpha = 0.85$ | estructura (SSIM) + fidelidad de intensidad/color (L1) |

L1 y L2 son `torch.nn.L1Loss` / `torch.nn.MSELoss`; SSIM y SSIM + L1 están en `src/losses.py` (con `torchmetrics`).

Para ver qué "le importa" a cada pérdida, la celda siguiente las evalúa entre un batch real y tres versiones alteradas: ruido gaussiano, desenfoque y un cambio de brillo.

In [ ]:
# Comparación de las 4 pérdidas ante distintas alteraciones
import torch.nn.functional as F

LOSS_NAMES = ["l1", "l2", "ssim", "ssim_l1"]
loss_fns = {name: instantiate(load_cfg([f"loss={name}"]).loss.fn) for name in LOSS_NAMES}

xb = x[:16]
perturbed = {
    "idéntica": xb,
    "ruido gaussiano (σ = 0.05)": (xb + 0.05 * torch.randn_like(xb)).clamp(0, 1),
    "desenfoque (promedio 5x5)": F.avg_pool2d(xb, 5, stride=1, padding=2, count_include_pad=False),
    "brillo + 0.1": (xb + 0.1).clamp(0, 1),
}
loss_table = pd.DataFrame({name: {loss: fn(p, xb).item() for loss, fn in loss_fns.items()}
                           for name, p in perturbed.items()}).T
display(loss_table.round(4))

**Lectura preliminar (completar con la salida):** se espera que L1/L2 castiguen sobre todo el cambio de brillo (desplaza todos los píxeles) y casi nada el desenfoque, mientras que SSIM castigue el ruido y el desenfoque (cambian la estructura local) y muy poco el brillo. Las escalas son distintas, por lo que **los valores de `loss` de corridas con distintas pérdidas no se comparan directamente**; para eso se registran también `l1`, `mse` y `ssim` en todas las corridas.

## 6. Entrenamiento
Se entrenan **8 corridas**: 2 modelos × 4 funciones de pérdida, con **los mismos hiperparámetros** (solo cambia la pérdida dentro de cada modelo). Todo se controla con Hydra: `train.py` compone la configuración, construye datos, modelo, pérdida, logger y `Trainer`, entrena con `trainer.fit` y evalúa el mejor checkpoint (menor `val/loss`) con `trainer.test`.

Hiperparámetros comunes (de `conf/`):

In [ ]:
# Hiperparámetros comunes a las 8 corridas
hparams = {
    "épocas": cfg.trainer.max_epochs,
    "batch size": cfg.data.batch_size,
    "tamaño de imagen": f"{cfg.data.img_size}x{cfg.data.img_size}x3",
    "validación": f"{cfg.data.val_split:.0%} de train/good por clase",
    "optimizador": f"Adam (lr = {cfg.model.optimizer.lr}, weight decay = {cfg.model.optimizer.weight_decay})",
    "dimensión latente z": cfg.model.net.latent_dim,
    "canales base": cfg.model.net.base_channels,
    "β (KL, solo VAE)": cfg.model.beta,
    "precisión": cfg.trainer.precision,
    "semilla": cfg.seed,
}
display(pd.Series(hparams, name="valor").to_frame())

Los entrenamientos se lanzan desde aquí ejecutando `train.py` como subproceso (Hydra necesita controlar su propio proceso), y su salida se muestra en vivo en la celda. Se desactiva la barra de progreso de tqdm (`trainer.enable_progress_bar=false`), que no se ve bien en las celdas, y en su lugar `EpochSummaryCallback` imprime una línea por época con `train/loss`, `val/loss` y `val/ssim`.

Cada corrida guarda su configuración, logs y checkpoints (`checkpoints/best.ckpt`, `last.ckpt`) en `outputs/<fecha>/<hora>_<corrida>/` (corrida simple) o `multirun/<fecha>/<hora>/<corrida>/` (multirun).

In [ ]:
# Función para lanzar train.py y ver su salida en la celda
import subprocess


def run_train(*overrides, multirun=False):
    # Run `python train.py [-m] <overrides>` with the .venv Python and stream its output into the cell
    cmd = [sys.executable, "train.py", *(["-m"] if multirun else []), "trainer.enable_progress_bar=false", *overrides]
    print(">>", " ".join(cmd[1:]), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONUTF8": "1", "HYDRA_FULL_ERROR": "1"}
    proc = subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace")
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:  # "Interrupt" in the notebook also stops training
        proc.terminate()
        raise
    if proc.wait() != 0:
        raise RuntimeError(f"train.py terminó con código {proc.returncode}")

### 6.1 Prueba corta (antes del entrenamiento completo)
2 épocas de cada modelo, con W&B en modo **offline** (no crea corridas en el proyecto). Sirve para comprobar que los datos, ambos modelos, las pérdidas SSIM, los callbacks de imágenes/t-SNE y el `test` funcionan antes de lanzar las 8 corridas, y para estimar cuánto tarda una época.

In [ ]:
# Prueba corta: 2 épocas por modelo (W&B offline)
run_train("trainer.max_epochs=2", "logger.wandb.mode=offline")                                 # vae + l1
run_train("model=u-net", "loss=ssim_l1", "trainer.max_epochs=2", "logger.wandb.mode=offline")  # u-net + ssim_l1

### 6.2 Entrenamiento completo: 8 corridas con Hydra multirun
`config.yaml` define el barrido en `hydra.sweeper.params`:

```yaml
hydra:
  sweeper:
    params:
      model: vae,u-net
      loss: l1,l2,ssim,ssim_l1
```

Con `-m`, Hydra ejecuta el producto cartesiano (8 trabajos) uno tras otro; cada uno es una corrida de W&B distinta (`vae-l1`, `vae-l2`, ..., `u-net-ssim_l1`). Equivale a `python train.py -m` en una terminal.

In [ ]:
# ENTRENAMIENTO COMPLETO: 8 corridas (2 modelos x 4 pérdidas)
run_train(multirun=True)

### 6.3 (Opcional) Repetir una sola corrida
Si alguna corrida falla o se quiere repetir, se puede lanzar sola con los mismos hiperparámetros. Para comparar, en la sección 7 se toma la corrida **más reciente** de cada combinación modelo/pérdida.

In [ ]:
# (Opcional) Repetir una sola corrida
# run_train("model=u-net", "loss=ssim")

## 7. Seguimiento de los entrenamientos en W&B
> **Base preliminar:** las celdas de las secciones 7-9 se escribieron antes de tener las corridas; se ajustarán cuando existan.

Desde aquí todo se obtiene de W&B con su API (`src/wandb_utils.py`): curvas, imágenes y figuras de t-SNE registradas durante el entrenamiento, y la tabla de errores del set de prueba. Las figuras nuevas que se construyen con esos datos se registran en la corrida `notebook-analysis` antes de mostrarse.

In [ ]:
# Obtener las 8 corridas de W&B e iniciar la corrida de análisis
from src import wandb_utils as wu

runs = wu.get_runs(WANDB_PROJECT, WANDB_ENTITY)
for (model, loss), run in sorted(runs.items()):
    print(f"{model:<6} {loss:<8} {run.name:<16} {run.url}")
assert len(runs) == 8, f"Se esperaban 8 corridas terminadas, hay {len(runs)}"

analysis_run = start_notebook_run("notebook-analysis")

### 7.1 Pérdida de entrenamiento y validación
Para cada modelo, las curvas de `train/loss` y `val/loss` de sus 4 corridas. Como cada corrida optimiza una pérdida distinta, las curvas sirven para ver **convergencia y sobreajuste** (separación train/val), no para decidir cuál es mejor; para eso, la segunda figura compara métricas comunes (`val/l1`, `val/mse`, `val/ssim`) en la misma escala.

In [ ]:
# Curvas de train/val loss por modelo
for model in wu.MODELS:
    log_and_show(analysis_run, f"curves/{model}_loss", fig=wu.loss_curves_figure(runs, model))

In [ ]:
# Métricas comunes de validación (L1, MSE, SSIM) por modelo
for model in wu.MODELS:
    fig = wu.loss_curves_figure(runs, model, keys=("val/l1", "val/mse", "val/ssim"))
    log_and_show(analysis_run, f"curves/{model}_common_metrics", fig=fig)

**Análisis (TODO):** velocidad de convergencia de cada pérdida, estabilidad, diferencia train/val, y qué pérdida logra mejor `val/ssim` y `val/l1` en cada modelo.

### 7.2 Reconstrucciones del set de validación (16 imágenes)
Imágenes registradas por `ReconstructionCallback` (filas impares: originales, filas pares: reconstrucciones), siempre las mismas 16 imágenes (4 por clase). Se muestran la primera y la última época de cada corrida.

In [ ]:
# Reconstrucciones de validación: primera y última época
for model in wu.MODELS:
    for loss in wu.LOSSES:
        history = wu.image_history(runs[(model, loss)], "val/reconstructions")
        for epoch, path in [history[0], history[-1]]:
            show_image(path, f"{model} / {loss} - época {epoch}")

**Análisis (TODO):** nitidez y fidelidad de color por modelo y pérdida (se espera VAE más borroso, U-Net casi idéntica a la entrada), y clases más difíciles (p. ej. *screw*, *cable*).

### 7.3 Espacio latente con t-SNE (validación, a lo largo de las épocas)
`LatentTSNECallback` proyecta con t-SNE el vector latente de todas las imágenes de validación cada 10 épocas, coloreado por clase. Primero la evolución de una corrida, luego la última época de todas.

In [ ]:
# Evolución del t-SNE de validación de una corrida
TSNE_RUN = ("vae", "ssim_l1")  # run whose evolution is shown

for step, fig in wu.plotly_history(runs[TSNE_RUN], "val/tsne"):
    fig.show()

In [ ]:
# t-SNE de validación de la última época de cada corrida
for model in wu.MODELS:
    for loss in wu.LOSSES:
        fig = wu.last_plotly(runs[(model, loss)], "val/tsne")
        fig.update_layout(title=f"{model} / {loss} - " + (fig.layout.title.text or ""))
        fig.show()

**Análisis (TODO):** si las 4 clases se separan en grupos, desde qué época, y diferencias entre el latente del VAE (regularizado por el KL) y el de la U-Net (promedio del cuello de botella).

## 8. Evaluación del modelo
La evaluación usa el set de prueba completo con el mejor checkpoint de cada corrida. El **error de reconstrucción** de cada imagen se usa como puntaje de anomalía: si el modelo solo aprendió lo normal, las imágenes defectuosas deberían tener mayor error.

### 8.1 Reconstrucción de imágenes buenas y con anomalías (16 imágenes de prueba)
`TestReconstructionCallback`: filas 1-3 son 8 imágenes buenas (original, reconstrucción, mapa de error $|x - \hat{x}|$) y filas 4-6 son 8 defectuosas. El mapa de error comparte la misma escala en ambas mitades.

In [ ]:
# Reconstrucciones del set de prueba (good vs defectuosas)
for model in wu.MODELS:
    for loss in wu.LOSSES:
        show_image(wu.last_image(runs[(model, loss)], "test/reconstructions"), f"{model} / {loss} - set de prueba")

t-SNE del set de prueba (verde = good, rojo = defective; el símbolo indica la clase):

In [ ]:
# t-SNE del set de prueba (good vs defective)
for model in wu.MODELS:
    for loss in wu.LOSSES:
        fig = wu.last_plotly(runs[(model, loss)], "test/tsne")
        fig.update_layout(title=f"{model} / {loss} - " + (fig.layout.title.text or ""))
        fig.show()

**Análisis (TODO):** ¿el mapa de error se concentra en el defecto?, ¿la reconstrucción "corrige" el defecto (deseable) o lo copia (U-Net)?, ¿se separan good y defective en el latente?

### 8.2 Histogramas del error de reconstrucción
A partir de la tabla `test/scores` (error por imagen) se construyen, por clase, histogramas del error de las imágenes **good** contra **cada tipo de defecto**. Cuanto menos se solapen, mejor separa el modelo ese defecto. Se grafican para la mejor corrida de cada modelo (mayor AUROC); cambiando `HIST_RUNS` y `SCORE` se pueden ver las demás.

In [ ]:
# Histogramas del error: good vs cada tipo de defecto
SCORE = "score_l1"  # score_l1 | score_mse | score_ssim
auroc_key = "test/auroc_" + SCORE.removeprefix("score_")

best = {model: max(wu.LOSSES, key=lambda l: runs[(model, l)].summary.get(auroc_key, 0)) for model in wu.MODELS}
HIST_RUNS = [(model, loss) for model, loss in best.items()]
print("Corridas para los histogramas:", HIST_RUNS)

scores = {key: wu.load_table(run) for key, run in runs.items()}  # test/scores of every run (used again in 9)

for model, loss in HIST_RUNS:
    for category in CLASSES:
        fig = wu.error_histograms(scores[(model, loss)], category, SCORE, title_prefix=f"{model} / {loss} - ")
        log_and_show(analysis_run, f"histograms/{model}_{loss}/{category}", fig=fig)

**Análisis (TODO):** qué defectos se detectan bien (defectos grandes: `missing_cable`, `misplaced`, ...) y cuáles se solapan con las imágenes buenas (defectos pequeños: `scratch_neck`, `poke`, ...).

## 9. Comparación entre configuraciones
Resumen de las 8 corridas con métricas en la misma escala:
- `val/*`: valor de la última época en validación (solo imágenes buenas).
- `test/l1`, `test/ssim`: calidad de reconstrucción en todo el set de prueba (mejor checkpoint).
- `test/auroc_*`: qué tan bien el error de reconstrucción separa good de defective (0.5 = azar, 1 = perfecto).

In [ ]:
# Tabla comparativa de las 8 corridas
keys = ["val/l1", "val/mse", "val/ssim", "test/l1", "test/ssim", "test/auroc_l1", "test/auroc_mse", "test/auroc_ssim"]
comparison = pd.DataFrame(wu.summary_table(runs, keys)).set_index(["model", "loss"]).drop(columns="run")
analysis_run.log({"comparison/summary": wandb.Table(dataframe=comparison.reset_index())})
display(comparison.round(4))

In [ ]:
# AUROC por clase, modelo y pérdida
import plotly.express as px

per_class = pd.DataFrame(wu.summary_table(runs, [f"{auroc_key}/{c}" for c in CLASSES])).drop(columns="run")
per_class = per_class.melt(id_vars=["model", "loss"], var_name="clase", value_name="auroc")
per_class["clase"] = per_class["clase"].str.split("/").str[-1]

fig = px.bar(per_class, x="clase", y="auroc", color="loss", barmode="group", facet_col="model",
             color_discrete_map=wu.LOSS_COLORS, title=f"AUROC por clase ({SCORE})", template="plotly_white")
fig.add_hline(y=0.5, line_dash="dot")
log_and_show(analysis_run, "comparison/auroc_per_class", fig=fig)

Error medio de reconstrucción en imágenes **correctas** vs **defectuosas** por configuración. Un buen detector tiene error bajo en las buenas y una razón defectuosas/buenas alta.

In [ ]:
# Error medio good vs defective por configuración
rows = []
for (model, loss), table in sorted(scores.items()):
    for category, (mean_good, mean_bad) in wu.good_vs_defective_means(table, SCORE).items():
        rows.append({"model": model, "loss": loss, "clase": category,
                     "error good": mean_good, "error defective": mean_bad, "razón def/good": mean_bad / mean_good})
errors = pd.DataFrame(rows)
display(errors[errors.clase == "all"].set_index(["model", "loss"]).drop(columns="clase").round(4))

long = errors[errors.clase == "all"].melt(id_vars=["model", "loss"], value_vars=["error good", "error defective"],
                                          var_name="tipo", value_name="error")
fig = px.bar(long, x="loss", y="error", color="tipo", barmode="group", facet_col="model",
             color_discrete_map={"error good": "#2ca02c", "error defective": "#d62728"},
             title=f"Error medio de reconstrucción ({SCORE}): good vs defective", template="plotly_white")
log_and_show(analysis_run, "comparison/good_vs_defective", fig=fig)

**Análisis (TODO):** mejor modelo/pérdida para *reconstruir* (val/test SSIM, L1) vs mejor para *detectar anomalías* (AUROC); no tienen por qué coincidir. Relacionar con las reconstrucciones de 7.2 y 8.1 y con los histogramas de 8.2.

In [ ]:
# Cerrar la corrida de análisis en W&B
analysis_run.finish()

## 10. Conclusiones
TODO (completar con los resultados). Puntos a cubrir:
1. **VAE vs U-Net:** calidad de reconstrucción vs capacidad de detectar defectos; efecto de las *skip connections*.
2. **Funciones de pérdida:** cuál converge mejor, cuál da reconstrucciones más nítidas y cuál separa mejor good/defective (L1 vs L2 vs SSIM vs SSIM + L1).
3. **Por clase y por defecto:** qué objetos y tipos de defecto son fáciles o difíciles, según los histogramas.
4. **Espacio latente:** qué muestra el t-SNE sobre lo aprendido (clases bien separadas, good vs defective).
5. **Limitaciones y mejoras:** resolución 128x128 (defectos pequeños se pierden), pocas épocas/datos, β del VAE, usar SSIM por regiones o mapas de error para localizar defectos.